In [1]:
# ================= HYPERTENSION — FINAL TRAINED & COMPRESSED MODEL  =================
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, mutual_info_classif
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, average_precision_score   # ← mAP added

from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.combine import SMOTETomek
from imblearn.ensemble import BalancedRandomForestClassifier

from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from xgboost import XGBClassifier

df = pd.read_csv("hypertension.csv")

y = df["TenYearCHD"].astype(int)
X = pd.get_dummies(df.drop(columns=["TenYearCHD"]), drop_first=True)
X_cols = X.columns.tolist()

X = SimpleImputer(strategy="median").fit_transform(X)

balancing = {
    "SMOTE": SMOTE(random_state=42),
    "ADASYN": ADASYN(random_state=42),
    "TOMEK": SMOTETomek(random_state=42)
}

models = {
    "LightGBM": LGBMClassifier(class_weight="balanced"),
    "CatBoost": CatBoostClassifier(verbose=0),
    "XGBoost": XGBClassifier(eval_metric="logloss"),
    "BalancedRF": BalancedRandomForestClassifier(n_estimators=600)
}

best_name = None
best_auc  = -1
best_model = None
best_selector = None
best_metrics = {}

print("\n=== HYPERTENSION MODEL OPTIMIZATION ===\n")

for bal_name, resampler in balancing.items():
    X_res, y_res = resampler.fit_resample(X, y)
    Xtr, Xte, ytr, yte = train_test_split(X_res, y_res, test_size=0.2, stratify=y_res, random_state=42)

    selector = SelectKBest(mutual_info_classif, k=min(25, Xtr.shape[1]))
    Xtr2 = selector.fit_transform(Xtr, ytr)
    Xte2 = selector.transform(Xte)

    print(f"\n--- [{bal_name}] ---")
    for model_name, model in models.items():
        model.fit(Xtr2, ytr)

        prob = model.predict_proba(Xte2)[:, 1]
        pred = (prob > 0.5).astype(int)

        acc = accuracy_score(yte, pred)
        f1 = f1_score(yte, pred)
        auc = roc_auc_score(yte, prob)
        mAP = average_precision_score(yte, prob)          # ← mAP added

        print(f"{model_name:<12} | ACC={acc:.4f} | F1={f1:.4f} | AUC={auc:.4f} | mAP={mAP:.4f}")

        if auc > best_auc:
            best_auc = auc
            best_name = f"{model_name} + {bal_name}"
            best_model = model
            best_selector = selector
            best_metrics = {"ACC":acc,"F1":f1,"AUC":auc,"mAP":mAP}

# ======================= PRINT RESULT ONLY =======================
print("\n====================================================")
print(f"🔥 BEST HYPERTENSION MODEL → {best_name}")
print(f"📌 AUC = {best_metrics['AUC']:.4f}")
print(f"💥 mAP = {best_metrics['mAP']:.4f}")    # ← REQUIRED OUTPUT
print("====================================================\n")



=== HYPERTENSION MODEL OPTIMIZATION ===


--- [SMOTE] ---
[LightGBM] [Info] Number of positive: 2877, number of negative: 2876
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000685 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3249
[LightGBM] [Info] Number of data points in the train set: 5753, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
LightGBM     | ACC=0.9013 | F1=0.8960 | AUC=0.9455 | mAP=0.9605


C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


CatBoost     | ACC=0.8999 | F1=0.8935 | AUC=0.9443 | mAP=0.9593
XGBoost      | ACC=0.8916 | F1=0.8868 | AUC=0.9437 | mAP=0.9590
BalancedRF   | ACC=0.9117 | F1=0.9083 | AUC=0.9668 | mAP=0.9742

--- [ADASYN] ---
[LightGBM] [Info] Number of positive: 2910, number of negative: 2877
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000318 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3188
[LightGBM] [Info] Number of data points in the train set: 5787, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
LightGBM     | ACC=0.8887 | F1=0.8831 | AUC=0.9456 | mAP=0.9598


C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


CatBoost     | ACC=0.8839 | F1=0.8754 | AUC=0.9419 | mAP=0.9565
XGBoost      | ACC=0.8818 | F1=0.8772 | AUC=0.9443 | mAP=0.9586
BalancedRF   | ACC=0.9026 | F1=0.8978 | AUC=0.9682 | mAP=0.9745

--- [TOMEK] ---
[LightGBM] [Info] Number of positive: 2853, number of negative: 2852
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000608 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3243
[LightGBM] [Info] Number of data points in the train set: 5705, number of used features: 15
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000
LightGBM     | ACC=0.9047 | F1=0.8990 | AUC=0.9551 | mAP=0.9658


C:\Users\Dell\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


CatBoost     | ACC=0.9026 | F1=0.8963 | AUC=0.9534 | mAP=0.9647
XGBoost      | ACC=0.9033 | F1=0.8991 | AUC=0.9512 | mAP=0.9646
BalancedRF   | ACC=0.9222 | F1=0.9204 | AUC=0.9717 | mAP=0.9774

🔥 BEST HYPERTENSION MODEL → BalancedRF + TOMEK
📌 AUC = 0.9717
💥 mAP = 0.9774

